# xla

> Training on TPUs and on AWS Trainium, which compile the model's graph with XLA: batches of fixed shapes, so a graph compiles once and is reused, and the presets for both; and the export of a model for Inferentia2.

In [ ]:
#| default_exp xla

In [ ]:
#| export
import json, os, sys, warnings
from pathlib import Path

import torch
from fastcore.basics import patch

from sysone.core import *
from sysone.data import DecisionCollator
from sysone.models import SideEncoder
from sysone.learner import Learner

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_ne
import logging, tempfile
logging.getLogger("transformers").setLevel(logging.ERROR)
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()
from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.inference import PositionScores, Decider
from sysone.cloud import remote

On a GPU, every batch can have its own shape: rows are padded to the longest row of the batch, and to the most options any of its questions has. A TPU, or a Trainium chip, runs the model through XLA, which compiles a graph for each shape it meets: a new shape is a new compilation, which takes seconds to minutes. So on XLA the batches must take few shapes:

- **lengths in buckets**: a batch is padded to the smallest of a few lengths (128, 256, 512, ... up to the template's `max_len`) that holds its longest row, so a training run meets a handful of shapes, not one per batch;
- **a fixed number of options**: every row is padded to the most options of any question in the data;
- **full batches**: the last, smaller batch of an epoch is dropped (`dataloader_drop_last`);
- **no decisions on the host in the step**: anything that turns a tensor into a Python value inside the training step (`bool(t.any())`, `t.item()`, indexing with a mask whose count changes) stops the graph and makes the next part's shape depend on the data. The loss skips rows with no answer by indexing, which is such a step; when every training row has an answer, as is usual, `prepare_xla` tells the trainer so, and the loss runs on the whole batch. Side encoders, which skip repeated inputs by comparing them on the host, encode every row.

`prepare_xla(learn)` does all of this, and a learner calls it by itself with the `tpu` or `neuron` preset (which a job on a TPU or a Neuron machine sets, see [cloud](40_cloud.ipynb)). On PyTorch/XLA, the Trainer moves the model to the XLA device itself; bf16 is the presets' precision. PyTorch/XLA is being succeeded on TPUs by TorchTPU (public from October 2026) and on Trainium by a native Neuron backend; both keep the need for few shapes, which is what this notebook provides.

In [ ]:
data = TypedDecisions.from_json("fixtures/typed_decisions_tiny.jsonl", valid=0.25, calib=0.2)
learn = decision_learner(data, "fixtures/tiny-modernbert", train="head", preset="cpu", logging_steps=1, disable_tqdm=True)

## Shapes

In [ ]:
#| export
def length_buckets(max_len:int, smallest:int=64) -> tuple:
    "The lengths a batch is padded to: powers of two from `smallest`, and `max_len` itself"
    out, n = [], smallest
    while n < max_len: out.append(n); n *= 2
    return tuple(out) + (max_len,)

def bucket(n:int, buckets:tuple) -> int:
    "The smallest bucket that holds `n` (the largest, when none does)"
    return next((b for b in buckets if b >= n), buckets[-1])

OPTION_KEYS = ("marker_pos", "marker_mask", "target", "spans")

def _grow(t, size:int, axis:int, value):
    if t.shape[axis] >= size: return t
    pad = list(t.shape); pad[axis] = size - t.shape[axis]
    return torch.cat([t, torch.full(pad, value, dtype=t.dtype)], axis)

def pad_batch(b:dict, L:int, K:int, pad_id:int=0, sides:dict|None=None) -> Batch:
    "Pad a collated batch: the sequence (and extras that follow it) to length `L`, the options to `K`, each side stream to its length in `sides`"
    l, k = b["input_ids"].shape[1], b["marker_pos"].shape[1]
    if l > L or k > K: raise ValueError(f"a batch of length {l} with {k} options does not fit {L} and {K}")
    out = dict(b)
    out["input_ids"], out["attention_mask"] = _grow(b["input_ids"], L, 1, pad_id), _grow(b["attention_mask"], L, 1, 0)
    for key, how in EXTRA_COLLATE.items():
        kind, value = how if isinstance(how, tuple) else (how, 0)
        if key in b and kind == "pad": out[key] = _grow(b[key], L, -1, value)
    for key in OPTION_KEYS:
        if key in b: out[key] = _grow(b[key], K, 1, 0)
    for name, S in (sides or {}).items():
        for key in (f"{name}_input_ids", f"{name}_attention_mask"):
            if key not in b: continue
            if b[key].shape[1] > S: raise ValueError(f"side stream {name} has {b[key].shape[1]} tokens, more than {S}")
            how = EXTRA_COLLATE.get(key, ("pad_rows", 0))
            out[key] = _grow(b[key], S, 1, how[1] if isinstance(how, tuple) else 0)
    return Batch(out)

class StaticCollator(DecisionCollator):
    "A collator for XLA: rows padded to a length bucket and a fixed number of options, side streams to buckets of their own"
    def __init__(self, pad_id:int=0, buckets:tuple=(512,), options:int=8, side_buckets:dict|None=None):
        super().__init__(pad_id); self.buckets, self.options, self.side_buckets = tuple(buckets), options, dict(side_buckets or {})
    def __call__(self, rows):
        b = super().__call__(rows)
        sides = {n: bucket(b[f"{n}_input_ids"].shape[1], bs) for n, bs in self.side_buckets.items() if f"{n}_input_ids" in b}
        return pad_batch(b, bucket(b["input_ids"].shape[1], self.buckets), self.options, self.pad_id, sides)
    def __repr__(self): return f"StaticCollator(pad_id={self.pad_id}, buckets={self.buckets}, options={self.options}, sides={self.side_buckets})"

In [ ]:
test_eq(length_buckets(512), (64, 128, 256, 512))
test_eq(length_buckets(600, 128), (128, 256, 512, 600))
test_eq([bucket(n, (64, 128, 256)) for n in (10, 64, 65, 300)], [64, 64, 128, 256])
rows = learn.dataset("train")
col = StaticCollator(learn.data.builder.tok.pad_token_id, length_buckets(512), options=6)
shapes = {tuple(col([rows[i] for i in range(j, j + 4)])[k].shape) for j in range(0, 8, 4) for k in ("input_ids", "marker_pos")}
assert all(s[1] in (64, 128, 256, 512, 6) for s in shapes)
bb = col([rows[i] for i in range(4)])
test_eq((bb["marker_mask"].shape[1], bb["target"].shape[1], bb["attention_mask"].shape), (6, 6, bb["input_ids"].shape))

## prepare_xla

In [ ]:
#| export
def max_options(learn) -> int:
    "The most options of any question in the learner's training and validation rows"
    k = 0
    for split in ("train", "valid"):
        if learn.data.cases.get(split) is None: continue
        ds = learn.dataset(split)
        col = ds["markers"] if hasattr(ds, "column_names") else [r["markers"] for r in ds]
        k = max(k, max((len(m) for m in col), default=0))
    return k

def all_answerable(learn) -> bool:
    "Whether every training row has an answer (a target with some mass)"
    ds = learn.dataset("train")
    ts = ds["target"] if hasattr(ds, "column_names") else [r["target"] for r in ds]
    return all(t is not None and sum(t) > 0 for t in ts)

def side_lengths(learn) -> dict:
    "The longest input of each side stream in the learner's training and validation rows"
    out = {}
    for name in learn.data.builder.streams:
        for split in ("train", "valid"):
            if learn.data.cases.get(split) is None: continue
            ds, key = learn.dataset(split), f"{name}_input_ids"
            col = ds[key] if hasattr(ds, "column_names") and key in ds.column_names else [r.get(key) or [] for r in ds]
            out[name] = max(out.get(name, 0), max((len(x) for x in col), default=0))
    return out

def prepare_xla(learn:Learner, buckets:tuple|None=None, options:int|None=None) -> Learner:
    "Make a learner's batches and step suit XLA: length buckets, fixed options, the loss on whole batches, side encoders without dedup"
    tmpl = learn.data.builder.template
    buckets = buckets or length_buckets(tmpl.max_len)
    sides = {n: length_buckets(L, 64) for n, L in side_lengths(learn).items() if L}
    col = StaticCollator(learn.data.builder.tok.pad_token_id or 0, buckets, options or max_options(learn), sides)
    learn.trainer.data_collator = col
    learn.trainer.all_answerable = all_answerable(learn)
    if not learn.trainer.all_answerable: warnings.warn("some training rows have no answer: the loss indexes them out, which recompiles on XLA")
    for m in learn.model.modules():
        if isinstance(m, SideEncoder): m.dedupe = False
    learn.xla = {"buckets": list(buckets), "options": col.options, "all_answerable": learn.trainer.all_answerable}
    return learn

def xla_metrics() -> dict:
    "PyTorch/XLA's compilation counters (how many graphs were compiled, and how long that took), when torch_xla is loaded"
    if "torch_xla" not in sys.modules: return {}
    import torch_xla.debug.metrics as met
    d = met.metric_data("CompileTime")
    return {"compiles": d[0] if d else 0, "compile_seconds": round(d[1] / 1e9, 2) if d else 0.0}

In [ ]:
prepare_xla(learn)
test_eq((learn.xla["options"] >= 2, learn.xla["all_answerable"], type(learn.trainer.data_collator).__name__), (True, True, "StaticCollator"))
before = {k: v.clone() for k, v in learn.model.head.state_dict().items()}
learn.fit(1, lr=1e-3)                         # the same training, on fixed shapes
assert any(not torch.equal(v, learn.model.head.state_dict()[k]) for k, v in before.items())
learn.xla

The step that runs here is the step XLA compiles: on the TPU runtime, `PJRT_DEVICE=TPU` and the `tpu` preset make the Trainer place the model on the TPU, and the batches above keep the number of compilations to the number of length buckets the data reaches. The check below runs the same training on PyTorch/XLA's CPU device in a Linux container (torch_xla has no macOS build), and counts the compilations of the first and second epochs: the second epoch must compile nothing new. On the fixtures, the first epoch compiled 8 graphs (7.9 s, under amd64 emulation) and the second none.

::: {.callout-note title="Finding: transformers' default optimizer has no XLA kernel"}
On the XLA device, the first training step stopped at the optimizer: transformers 5 uses PyTorch's fused AdamW by default (`optim="adamw_torch_fused"`), which runs on CUDA, MPS, CPU and a few other devices, not on XLA. The `tpu` and `neuron` presets use the unfused AdamW (`optim="adamw_torch"`). The forward and backward passes had already run on XLA, so this was the last thing between sysone and a TPU.
:::

In [ ]:
#| export
XLA_CHECK = r'''import json, os, sys
os.environ.setdefault("PJRT_DEVICE", "CPU")
import torch_xla
from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.xla import prepare_xla, xla_metrics
data = TypedDecisions.from_json(sys.argv[1], valid=0.25, calib=0.2)
learn = decision_learner(data, sys.argv[2], train="head", preset="tpu", logging_steps=1, disable_tqdm=True, bf16=False, calibrate_after_fit=False)
learn.fit(1, lr=1e-3); first = xla_metrics()
learn.fit(1, lr=1e-3); second = xla_metrics()
print("SYSONE-XLA " + json.dumps({"first": first, "second": second, "device": str(next(learn.model.parameters()).device), "xla": learn.xla}))
'''

def xla_check_image(tag:str="sysone-xla-check", platform:str="linux/amd64") -> str:
    "Build a Linux image with torch_xla's CPU device and this sysone, to check that XLA trains with few compilations"
    import subprocess, tempfile
    from sysone.cloud import build_wheel
    ctx = Path(tempfile.mkdtemp(prefix="sysone-xla-")); whl = build_wheel(ctx / "wheels")
    (ctx / "Dockerfile").write_text("FROM python:3.12-slim\nENV PYTHONUNBUFFERED=1 PIP_NO_CACHE_DIR=1\n"
        'RUN pip install --index-url https://download.pytorch.org/whl/cpu "torch==2.9.0" && pip install "torch_xla==2.9.0"\n'
        "COPY wheels/ /w/\nRUN pip install /w/*.whl\n")
    r = subprocess.run(["docker", "build", "--platform", platform, "-t", tag, str(ctx)], capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(f"docker build failed: {r.stdout[-1500:]}{r.stderr[-1500:]}")
    return tag

def xla_check(data, encoder, tag:str="sysone-xla-check", platform:str="linux/amd64") -> dict:
    "Train a learner on PyTorch/XLA's CPU device in a container, two epochs; returns the compilations of each"
    import subprocess, tempfile
    work = Path(tempfile.mkdtemp(prefix="sysone-xla-run-"))
    (work / "check.py").write_text(XLA_CHECK)
    import shutil
    shutil.copy2(data, work / Path(data).name); shutil.copytree(encoder, work / "encoder")
    r = subprocess.run(["docker", "run", "--rm", "--platform", platform, "-v", f"{work}:/w", "-w", "/w", tag,
                        "python", "check.py", Path(data).name, "encoder"], capture_output=True, text=True)
    line = next((l for l in r.stdout.splitlines() if l.startswith("SYSONE-XLA ")), None)
    if line is None: raise RuntimeError(f"the XLA check failed: {(r.stdout + r.stderr)[-2500:]}")
    return json.loads(line[len("SYSONE-XLA "):])

In [ ]:
#| xla
tag = xla_check_image()
res = xla_check("fixtures/typed_decisions_tiny.jsonl", "fixtures/tiny-modernbert", tag)
assert res["device"].startswith("xla"), res
test_eq(res["second"]["compiles"], res["first"]["compiles"])     # the second epoch reuses every graph of the first
res

### On a Colab TPU

Colab's TPU VMs (a v5e-1, which the free tier offers) come with PyTorch/XLA. In October 2026 that was torch 2.9 with torch_xla 2.9 and `libtpu`, with `PJRT_DEVICE=TPU` set, beside the same transformers as here, so a [job](45_colab.ipynb) on one installs sysone alone. This cell, flagged `colab`, runs the check above as a job on a v5e-1: the `tpu` preset, which a job on a TPU sets, puts the model on the TPU, the second epoch must compile nothing new, and the export, trained there, answers here.

In [ ]:
#| colab
os.environ |= {"SYSONE_JOBS": str(Path(tempfile.mkdtemp()) / "jobs.json"), "SYSONE_JOBS_DIR": tempfile.mkdtemp()}
tpu_script = Path(tempfile.mkdtemp()) / "train_tpu.py"
tpu_script.write_text("""import json, os
from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.cloud import job_input, job_output
from sysone.xla import xla_metrics
data = TypedDecisions.from_json(job_input("data"), valid=0.25, calib=0.2)
learn = decision_learner(data, os.environ["ENCODER"], train="head", logging_steps=1, disable_tqdm=True)
learn.fit(1, lr=1e-3); first = xla_metrics()
learn.fit(1, lr=1e-3); second = xla_metrics()
print("SYSONE-XLA " + json.dumps({"first": first, "second": second, "device": str(next(learn.model.parameters()).device), "preset": learn.preset}), flush=True)
learn.export(job_output())
""")
tj = remote(tpu_script, on="colab", hw="tpu", data="fixtures/typed_decisions_tiny.jsonl", env={"ENCODER": "answerdotai/ModernBERT-base"})
tj.wait(every=20)
test_eq(tj.status, "complete")
tres = json.loads(next(l for l in tj.logs().splitlines() if l.startswith("SYSONE-XLA "))[len("SYSONE-XLA "):])
test_eq((tres["device"].startswith("xla"), tres["preset"]), (True, "tpu"))
test_eq(tres["second"]["compiles"], tres["first"]["compiles"])      # on the TPU too, the second epoch compiles nothing new
rec = json.loads(Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()[0])
test_eq(set(Decider.load(tj.fetch(), device="cpu").predict(rec["state"], rec["questions"])), set(rec["questions"]))
tres

## Inferentia2

An Inferentia2 chip runs a model compiled ahead of time for fixed input shapes. `export_neuron` traces the part of a sysone model that runs per row, the encoder and the head's scoring of every position (the same `PositionScores` module the [ONNX export](06_inference.ipynb#onnx) uses), with `torch_neuronx.trace`, for a batch size, a length and a question kind; `NeuronScorer` runs it, padding each batch to those shapes. It needs the Neuron SDK (`torch-neuronx`, from AWS's package index), which runs on Linux only; on a machine without Neuron devices, `torch_neuronx.trace` compiles for a target named by `NEURON_PLATFORM_TARGET_OVERRIDE` (`inf2`, `trn1`, `trn2`), so the compile can be checked before any hardware is at hand.

In [ ]:
#| export
def neuron_inputs(learn, batch:int=1, length:int|None=None) -> tuple:
    "Example inputs of the fixed shapes a Neuron model is compiled for: token ids, attention mask, question kind"
    L = length or learn.data.builder.template.max_len
    return (torch.zeros(batch, L, dtype=torch.long), torch.ones(batch, L, dtype=torch.long), torch.zeros(batch, dtype=torch.long))

def export_neuron(learn, path, batch:int=1, length:int|None=None, target:str|None=None) -> Path:
    "Compile the encoder and the head's position scores for Inferentia2 or Trainium, at fixed shapes, beside an export in `path`"
    try: import torch_neuronx
    except ImportError: raise ImportError("export_neuron needs the Neuron SDK: pip install torch-neuronx --extra-index-url https://pip.repos.neuron.amazonaws.com") from None
    from sysone.inference import PositionScores
    path = Path(learn.export(path)) if not (Path(path) / "sysone.json").exists() else Path(path)
    if target: os.environ["NEURON_PLATFORM_TARGET_OVERRIDE"] = target
    mod = PositionScores(learn.model.float().eval().cpu())
    ex = neuron_inputs(learn, batch, length)
    traced = torch_neuronx.trace(mod, ex)
    torch.jit.save(traced, str(path / "model.neuron.pt"))
    meta = json.loads((path / "sysone.json").read_text())
    meta["neuron"] = {"file": "model.neuron.pt", "batch": batch, "length": ex[0].shape[1], "target": target or os.environ.get("NEURON_PLATFORM_TARGET_OVERRIDE")}
    (path / "sysone.json").write_text(json.dumps(meta, indent=2, default=str))
    return path

class NeuronScorer:
    "Run a model compiled by `export_neuron`: pads each batch to its compiled shapes"
    def __init__(self, path):
        import torch_neuronx  # noqa: F401  (registers the Neuron runtime with torch.jit)
        self.meta = json.loads((Path(path) / "sysone.json").read_text())["neuron"]
        self.model = torch.jit.load(str(Path(path) / self.meta["file"]))
    def __call__(self, input_ids, attention_mask, qtype):
        B, L = self.meta["batch"], self.meta["length"]
        n = input_ids.shape[0]
        if n > B or input_ids.shape[1] > L: raise ValueError(f"a batch of {n} rows of {input_ids.shape[1]} tokens exceeds the compiled {B} × {L}")
        pad = lambda t, v: torch.cat([torch.cat([t, torch.full((t.shape[0], L - t.shape[1]), v, dtype=t.dtype)], 1),
                                      torch.full((B - n, L), v, dtype=t.dtype)], 0)
        q = torch.cat([qtype, torch.zeros(B - n, dtype=qtype.dtype)])
        return self.model(pad(input_ids, 0), pad(attention_mask, 0), q)[:n, :input_ids.shape[1]]

In [ ]:
ids, att, qt = neuron_inputs(learn, batch=2, length=64)
test_eq((ids.shape, att.shape, qt.shape), ((2, 64), (2, 64), (2,)))
test_eq(PositionScores(learn.model.float().eval().cpu())(ids, att, qt).shape, (2, 64))      # the module export_neuron traces runs at these shapes
test_fail(lambda: export_neuron(learn, tempfile.mkdtemp()), contains="Neuron SDK")

As for XLA, the check runs in a Linux container: `neuron_check_image` adds the Neuron SDK (`torch-neuronx` and the `neuronx-cc` compiler, from AWS's package index) to the XLA image's PyTorch, and `neuron_check` builds the tiny model there and compiles it for Inferentia2 with `export_neuron`.

In [ ]:
#| export
NEURON_CHECK = r'''import json, os, sys
os.environ.setdefault("PJRT_DEVICE", "CPU")     # no Neuron device here: XLA's own device is the CPU; neuronx-cc compiles for the target
from pathlib import Path
from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.xla import export_neuron
data = TypedDecisions.from_json(sys.argv[1], valid=0.25, calib=0.2)
learn = decision_learner(data, sys.argv[2], train="head", preset="cpu", logging_steps=1, disable_tqdm=True, calibrate_after_fit=False)
out = export_neuron(learn, Path("/w/neuron-jev"), batch=1, length=int(sys.argv[3]), target=sys.argv[4])
meta = json.loads((out / "sysone.json").read_text())["neuron"]
print("SYSONE-NEURON " + json.dumps(meta | {"bytes": (out / meta["file"]).stat().st_size}))
'''

def neuron_check_image(tag:str="sysone-neuron-check", platform:str="linux/amd64") -> str:
    "Build a Linux image with the Neuron SDK's compiler and this sysone, to check that `export_neuron` compiles"
    import subprocess, tempfile
    from sysone.cloud import build_wheel
    ctx = Path(tempfile.mkdtemp(prefix="sysone-neuron-")); build_wheel(ctx / "wheels")
    (ctx / "Dockerfile").write_text("FROM python:3.12-slim\nENV PYTHONUNBUFFERED=1 PIP_NO_CACHE_DIR=1\n"
        'RUN pip install --index-url https://download.pytorch.org/whl/cpu "torch==2.9.0" && pip install "torch_xla==2.9.0"\n'
        'RUN pip install --extra-index-url https://pip.repos.neuron.amazonaws.com "torch-neuronx==2.9.*" neuronx-cc\n'
        'RUN pip install "islpy==2026.1"\n'      # neuronx-cc 2.27 asks for islpy~=2026.1, and 2026.2 changed a function it calls'
        # torch-neuronx loads the Neuron runtime (libnrt) even to compile, and the runtime comes from AWS's apt repository
        "RUN apt-get update && apt-get install -y --no-install-recommends gnupg wget ca-certificates && "
        "wget -qO - https://apt.repos.neuron.amazonaws.com/GPG-PUB-KEY-AMAZON-AWS-NEURON.PUB | gpg --dearmor > /usr/share/keyrings/neuron.gpg && "
        'echo "deb [signed-by=/usr/share/keyrings/neuron.gpg] https://apt.repos.neuron.amazonaws.com noble main" > /etc/apt/sources.list.d/neuron.list && '
        "apt-get update && apt-get install -y --no-install-recommends aws-neuronx-runtime-lib && "
        "(apt-get install -y --no-install-recommends libarchive13t64 || apt-get install -y --no-install-recommends libarchive13) && "   # the compiler's backend
        "rm -rf /var/lib/apt/lists/*\n"
        "ENV LD_LIBRARY_PATH=/opt/aws/neuron/lib\n"
        "COPY wheels/ /w/\nRUN pip install /w/*.whl\n")
    r = subprocess.run(["docker", "build", "--platform", platform, "-t", tag, str(ctx)], capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(f"docker build failed: {r.stdout[-1500:]}{r.stderr[-1500:]}")
    return tag

def neuron_check(data, encoder, length:int=64, target:str="inf2", tag:str="sysone-neuron-check", platform:str="linux/amd64") -> dict:
    "Train a learner in a Linux container and compile it for a Neuron target with `export_neuron`; returns the compiled model's record"
    import shutil, subprocess, tempfile
    work = Path(tempfile.mkdtemp(prefix="sysone-neuron-run-"))
    (work / "check.py").write_text(NEURON_CHECK)
    shutil.copy2(data, work / Path(data).name); shutil.copytree(encoder, work / "encoder")
    r = subprocess.run(["docker", "run", "--rm", "--platform", platform, "-v", f"{work}:/w", "-w", "/w", tag,
                        "python", "check.py", Path(data).name, "encoder", str(length), target], capture_output=True, text=True)
    line = next((l for l in r.stdout.splitlines() if l.startswith("SYSONE-NEURON ")), None)
    if line is None: raise RuntimeError(f"the Neuron check failed: {(r.stdout + r.stderr)[-3000:]}")
    return json.loads(line[len("SYSONE-NEURON "):])

In [ ]:
#| neuron
nres = neuron_check("fixtures/typed_decisions_tiny.jsonl", "fixtures/tiny-modernbert", tag=neuron_check_image())
test_eq((nres["target"], nres["length"], nres["batch"]), ("inf2", 64, 1))
assert nres["bytes"] > 0
nres

::: {.callout-note title="Finding: compiling for Inferentia2 without the hardware needed three things the Neuron SDK's packages do not bring"}
In a Linux container with `torch-neuronx` 2.9 and `neuronx-cc` 2.27 from AWS's package index, the tiny model compiled for `inf2` (a 1.5 MB `model.neuron.pt` for one row of 64 tokens), after three fixes:

- **the runtime library**: `torch-neuronx` loads `libnrt` even to compile, and it comes from AWS's apt repository (`aws-neuronx-runtime-lib`), not from pip; with it, torch_xla then picks the Neuron device by default, so `PJRT_DEVICE=CPU` keeps XLA's own work on the CPU while `neuronx-cc` compiles for the target;
- **islpy**: `neuronx-cc` 2.27 asks for `islpy~=2026.1`, pip takes 2026.2.2, and its changed `is_subset` makes the compiler fail with an internal error (`NCC_ISMP902`); `islpy==2026.1` works;
- **libarchive**: the compiler's backend needs the system library `libarchive.so.13`.

`neuron_check_image` does all three. The compiled model has not yet run on Inferentia2, which needs the hardware.
:::

On Trainium, training is the same as on a TPU: the Neuron SDK's PyTorch is PyTorch/XLA, so a job with `hw=Neuron("trn1.2xlarge")` on SageMaker runs with the `neuron` preset and `prepare_xla`. The Neuron SDK compiles each graph with `neuronx-cc`; running the job's script once under `neuron_parallel_compile` fills the compile cache ahead of time (`NEURON_COMPILE_CACHE_URL` can point the cache at S3, so jobs share it). Training on Trainium has not run yet; `export_neuron` compiles (checked above, for `inf2`), and running its output needs an Inferentia2 or Trainium machine.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()